# S4b — dip in uptrend as a TIME-SERIES EVENT STUDY on SPY + 11 sector SPDRs

Each ETF is tested against its own history. Pre-registered 2026-10-03, after pack 2's S4 was taken off real data
(binding protocol). The S4 tilt there came entirely from the 10% disaster-stop fill rule, so **there's no disaster
stop here**.

**Lockbox:** cut at load 2024-12-31; signal bars so that entries are 2011-01-03 .. 2023-12-29.

**Events:**
- Close > SMA200 and SMA50 > SMA200, and a trigger: T1 RSI(2) < 10 | T2 3 lower closes | T3 close < prior 20-bar low
  close.
- Entry at the next open. **Non-overlapping per ETF:** the next event is allowed only once the current window has
  ended.
- ETFs become eligible when SMA200 exists (XLRE ≈ 2016, XLC ≈ 2019).

**Outcomes:** 3 triggers × 5 = 15 TRIALS, pooled across the 12 ETFs.
- X1: close > SMA5. X2: RSI(2) > 70. Both exit at the next open, with a 10-bar time stop.
- Fixed horizons H5 / H10 / H20: open of t+1 → close of t+h.

**Excess per event** = event gross return − same-ETF baseline − 0.15%. The baseline is the mean return over a window
of the SAME length and exit price type, starting on every other uptrend day of that ETF where the same trigger did
not fire (IS range). Costs apply to the event only; the baseline is gross. Matching the length removes holding-time ×
drift.

**Statistics:** pooled mean excess and a month-block bootstrap 90% CI. q = share of bootstrap means > 0. Also break-
even (mean gross excess), per year, by era, and per ETF (descriptive).

**Verdict (pre-registered):** *worth drilling* iff ALL of:
- 2019-2023 mean excess > 0;
- ≥ 9 qualifying years (a year qualifies with ≥ 10 events) AND positive in ≥ 2/3 of the qualifying years;
- q ≥ 90.

**Martingale validation FIRST (standing rule; fail = STOP).** 16 worlds use the pack-2 generator restricted to the 12
ETFs: real listing spans, market drift on, intrabar paths, no edge.
- All on GROSS excess. The 0.15% cost is a known constant, so net excess is negative by construction; the verdict
  on real data uses NET.
- Per trigger family: mean gross excess over the 16 per-seed values with |t| < 2.5, and mean gross q in [40, 60]
  with |t| < 2.5.
- Pooled share of (seed × trial) with gross q ≥ 90 must be ≤ 15%.
- Power: +0.5%/bar planted for 3 bars after any > 2σ down close. T1 family mean gross q ≥ 90 in ≥ 3 of 4 seeds.

The real-data cells refuse to run unless `s4b/s4b_protocol.json` says PASS.

In [ ]:
# ===== Cell 0 — config (pack-2 base + S4b) =====
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass                                            # local smoke test: BUKOWSKI_DIR points at fixtures
import pandas as pd, numpy as np, glob, json, os, math, urllib.request
from pathlib import Path
from scipy import stats
import warnings; warnings.filterwarnings('ignore')

DATA_DIR = Path(os.environ.get('BUKOWSKI_DIR', '/content/drive/MyDrive/Bukowski'))
REF_DIR  = DATA_DIR/'reference'
OUT_DIR  = DATA_DIR/'results'/'screens'; OUT_DIR.mkdir(parents=True, exist_ok=True)

# ---- LOCKBOX ----
CUT_DATE = pd.Timestamp('2024-12-31')     # every series cut at load; nothing after this is read, pulled or printed
import re, io, zipfile, hashlib, time
from numba import njit, prange

IS_START, PURGE = pd.Timestamp('2011-01-01'), pd.Timestamp('2023-12-29')   # entries IS_START..PURGE; holds resolve <= cut
Y0, NY = 2011, 13                                         # years 2011..2023
ERAS = {'2011-2014': (2011, 2014), '2015-2018': (2015, 2018), '2019-2023': (2019, 2023)}; RECENT = '2019-2023'
COST = 0.0015                                             # round trip
TOP_U2 = 300                                              # U2 = top 300 stocks by trailing 60-bar $vol (t-60..t-1)
S4_HOLD, S4_STOP = 10, 0.10                               # 10-bar time stop; 10% disaster stop
S5_HOLDS = [20, 60]
S6_MIN_NAMES = 5                                          # top decile, at least 5 names
N_DRAWS, TRIES, DRAW_BATCH = int(os.environ.get('PACK2_DRAWS', 200)), 8, 50
PCT_PASS, MIN_YEARS = 90.0, 9                             # verdict: null pct >= 90; >= 9/13 years positive
FH_CANDIDATES = {'S4': [60], 'S5': [20, 60]}              # fixed-horizon excess (Stage 2b v2 method)
CI_LEVEL, BOOT_REPS = 0.90, 2000
P2_DIR = OUT_DIR/'pack2'; CACHE = P2_DIR/'cache'; CACHE.mkdir(parents=True, exist_ok=True)
ASSET_FILE, U1_FILE, PROTO_FILE = P2_DIR/'asset_class.csv', P2_DIR/'u1_etfs.csv', P2_DIR/'pack2_protocol.json'
UNIVS = ['U1', 'U2']
# Reviewed 2026-10-03 (Step 0 output): Tiingo rows misclassify these; shared by both notebooks.
ASSET_OVERRIDES = {'PAM': 'Stock',     # Pampa Energia SA (ADR), listed as ETF
                   'BNY': 'Stock',     # BNY Mellon (ticker from BK, 2024); old BNY row was a fund
                   'ERO': 'Stock'}     # Ero Copper; old ERO row was an exchange-traded note
CONFIGS = ([dict(name=f'S4_{tg}_{ex}_{st}', screen='S4', family=f'S4_{tg}', trig=tg, mode=(0 if ex == 'X1' else 1),
                 hold=S4_HOLD, stop=(st == 'stop10'))
            for tg in ['T1', 'T2', 'T3'] for ex in ['X1', 'X2'] for st in ['nostop', 'stop10']]
           + [dict(name=f'S5_h{h}', screen='S5', family='S5', trig=None, mode=2, hold=h, stop=False) for h in S5_HOLDS])


# ---- S4b (decisions 2026-10-03) ----
S4B_DIR = OUT_DIR/'s4b'; S4B_DIR.mkdir(parents=True, exist_ok=True); S4B_PROTO = S4B_DIR/'s4b_protocol.json'
ETFS = ['SPY', 'XLB', 'XLC', 'XLE', 'XLF', 'XLI', 'XLK', 'XLP', 'XLRE', 'XLU', 'XLV', 'XLY']
TRIGS = ['T1', 'T2', 'T3']
OUTCOMES = [('X1', 0, 10), ('X2', 1, 10), ('H5', 2, 5), ('H10', 2, 10), ('H20', 2, 20)]   # (name, sim mode, hold bars)
MIN_EVENTS_YEAR, MIN_QUAL_YEARS, Q_PASS = 10, 9, 90.0
S4B_SEEDS = int(os.environ.get('S4B_SEEDS', 16)); S4B_POWER_SEEDS = int(os.environ.get('S4B_POWER_SEEDS', 4))
M_STEPS, ON_FRAC, MKT_SD, MKT_MU = 12, 0.2, 0.01, 0.0003
PLANT_REV = 0.005                                         # power: +0.5%/bar for 3 bars after a > 2 sigma down close
TRIAL_LOG, TRIAL_RET = OUT_DIR/'screens_trial_log.csv', OUT_DIR/'screens_trial_returns.csv'

In [ ]:
# ===== Cell 1 — core (pack-2 core + S4b event study) =====
def load_px(path):
    """Read OHLC(+V) and CUT at CUT_DATE before anything else touches it."""
    cols = pd.read_csv(path, nrows=0).columns
    use = [c for c in ['Date', 'Open', 'High', 'Low', 'Close', 'Volume'] if c in cols]
    d = pd.read_csv(path, usecols=use); d['Date'] = pd.to_datetime(d['Date'])
    d = d.sort_values('Date').drop_duplicates('Date')
    d = d[d['Date'] <= CUT_DATE].reset_index(drop=True)                  # LOCKBOX
    if 'Volume' not in d.columns: d['Volume'] = np.nan
    return d


def tiingo_token():
    token = os.environ.get('TIINGO_API_KEY')
    if not token:
        try:
            from google.colab import userdata; token = userdata.get('TIINGO_API_KEY')
        except Exception:
            token = None
    return token


def ref_path(tk):
    """RSP historically lives at DATA_DIR/RSP.csv or REF_DIR/RSP.csv; everything new goes to REF_DIR."""
    return DATA_DIR/f'{tk}.csv' if tk == 'RSP' and (DATA_DIR/'RSP.csv').exists() else REF_DIR/f'{tk}.csv'


def ensure_ref(tk):
    """One-off ADJUSTED Tiingo pull, only if missing. endDate = CUT_DATE so sealed rows never reach Drive."""
    path = ref_path(tk)
    if path.exists():
        print(f'{tk}: present at {path} — no pull'); return path
    token = tiingo_token()
    assert token, f'Set TIINGO_API_KEY (Colab secret or env) to pull {tk}.'
    url = (f'https://api.tiingo.com/tiingo/daily/{tk}/prices?startDate=2003-01-01'
           f'&endDate={CUT_DATE:%Y-%m-%d}&token={token}')
    js = pd.DataFrame(json.load(urllib.request.urlopen(url, timeout=60)))
    df = pd.DataFrame({'Date': pd.to_datetime(js['date']).dt.strftime('%Y-%m-%d'),
                       'Open': js['adjOpen'], 'High': js['adjHigh'], 'Low': js['adjLow'],
                       'Close': js['adjClose'], 'Volume': js['adjVolume']})
    df = df[pd.to_datetime(df['Date']) <= CUT_DATE]                     # belt and braces
    REF_DIR.mkdir(parents=True, exist_ok=True); df.to_csv(path, index=False)
    print(f'{tk}: pulled {len(df)} rows (adjusted) -> {path}')
    return path


def close_ret(d, cal=None):
    """Close-to-close simple return on the series' own rows (optionally restricted to a calendar first)."""
    s = d.set_index('Date')['Close'].astype(float)
    if cal is not None: s = s[s.index.isin(cal)]
    s = s.where(s > 0)
    return (s / s.shift(1) - 1).dropna()


# ---------------------------------------------------------------- panels (T x N), cut at load, cached
def load_panels():
    f_cache = CACHE/'panels_cut_2024-12-31.npz'
    if f_cache.exists():
        z = np.load(f_cache)
        cal = pd.DatetimeIndex(z['dates']); assert cal.max() <= CUT_DATE
        return [str(t) for t in z['tickers']], cal, {c: z[c].astype(np.float64) for c in 'OHLCV'}
    spy = load_px(DATA_DIR/'SPY.csv'); cal = pd.DatetimeIndex(spy['Date']); T = len(cal)
    arrs, tickers = {c: [] for c in 'OHLCV'}, []
    for f in sorted(glob.glob(f'{DATA_DIR}/*.csv')):
        try:
            d = load_px(f)
        except Exception:
            continue
        pos = cal.get_indexer(d['Date']); ok = pos >= 0
        if not ok.any(): continue
        tickers.append(Path(f).stem.upper())
        for c, col in zip('OHLCV', ['Open', 'High', 'Low', 'Close', 'Volume']):
            a = np.full(T, np.nan, np.float32); a[pos[ok]] = d[col].to_numpy(float)[ok]; arrs[c].append(a)
    P = {c: np.stack(arrs[c], axis=1) for c in 'OHLCV'}
    bad = ~(P['C'] > 0)
    for c in 'OHLC': P[c][bad] = np.nan
    np.savez(f_cache, tickers=np.array(tickers), dates=cal.values, **P)
    return tickers, cal, {c: P[c].astype(np.float64) for c in P}


def lag(X, n):
    Y = np.full_like(X, np.nan); Y[n:] = X[:-n]; return Y


def roll(X, w, how='mean', minp=None):
    r = pd.DataFrame(X).rolling(w, min_periods=minp)
    return getattr(r, how)().to_numpy()


def indicators(O, H, L, C, V):
    """All causal: values at row t use bars <= t; the *p ones use bars <= t-1."""
    I = {'S5': roll(C, 5), 'S50': roll(C, 50), 'S200': roll(C, 200)}
    C1 = lag(C, 1); d = C - C1
    au = pd.DataFrame(np.where(d > 0, d, 0.0)).where(np.isfinite(d)).ewm(alpha=0.5, adjust=False).mean().to_numpy()
    ad = pd.DataFrame(np.where(d < 0, -d, 0.0)).where(np.isfinite(d)).ewm(alpha=0.5, adjust=False).mean().to_numpy()
    tot = au + ad
    I['RSI'] = np.where(np.isfinite(d), np.where(tot > 0, 100 * au / np.where(tot > 0, tot, 1), 50.0), np.nan)
    TR = np.fmax(H - L, np.fmax(np.abs(H - C1), np.abs(L - C1)))
    I['ATRp'] = lag(roll(TR, 14), 1)
    I['VA50p'] = lag(roll(V, 50), 1)
    I['DV60p'] = lag(roll(C * V, 60), 1)
    I['LOW20p'] = roll(C1, 20, 'min')
    ON = np.log(O / C1); ON[~(np.abs(ON) < np.log(1.5))] = np.nan          # overnight bad prints dropped
    I['ON21'] = roll(ON, 21, 'sum', minp=18)
    return I


def universes(DV60p, is_stock, is_u1):
    T, N = DV60p.shape
    X = np.where(is_stock[None, :], DV60p, np.nan)
    U2 = np.zeros((T, N), bool)
    for t in range(T):
        idx = np.nonzero(np.isfinite(X[t]) & (X[t] > 0))[0]
        if len(idx) > TOP_U2: idx = idx[np.argpartition(-X[t, idx], TOP_U2 - 1)[:TOP_U2]]
        U2[t, idx] = True
    U1 = is_u1[None, :] & np.isfinite(DV60p) & (DV60p > 0)
    return {'U1': U1, 'U2': U2}


def fwd(C, h):
    F = np.full_like(C, np.nan); F[:-h] = C[h:] / C[:-h] - 1; return F


def build_ctx(O, H, L, C, V, is_stock, is_u1, i_spy):
    I = indicators(O, H, L, C, V)
    UN = universes(I['DV60p'], is_stock, is_u1)
    C1, C2, C3 = lag(C, 1), lag(C, 2), lag(C, 3)
    trend = (C > I['S200']) & (I['S50'] > I['S200'])
    TRIG = {'T1': I['RSI'] < 10, 'T2': (C < C1) & (C1 < C2) & (C2 < C3), 'T3': C < I['LOW20p']}
    quiet = np.abs(C - C1) < I['ATRp']; spike = V >= 3 * I['VA50p']; va = I['VA50p'] > 0
    NT = {k: np.ascontiguousarray(x.T) for k, x in dict(O=O, H=H, L=L, C=C, S5=I['S5'], RSI=I['RSI']).items()}
    hs = sorted({h for v in FH_CANDIDATES.values() for h in v})
    FW = {h: fwd(C, h) for h in hs}
    EWF = {(u, h): np.nanmean(np.where(UN[u], FW[h], np.nan), axis=1) for u in UNIVS for h in hs}
    return dict(I=I, UN=UN, trend=trend, TRIG=TRIG, s5=spike & quiet & va, s5pool=quiet & ~spike & va,
                NT=NT, FW=FW, EWF=EWF, C=C, i_spy=i_spy)


# ---------------------------------------------------------------- simulator (Stage 2 semantics)
@njit(cache=True)
def sim_one(O, H, L, C, S5, RSI, k, e, mode, hold, use_stop, stop_pct):
    """Enter at O[k,e]. Bar order: disaster stop (low) -> time stop (close of bar e+hold-1) -> exit signal at the
    close (mode 0: close > SMA5; 1: RSI2 > 70; 2: none), filled at the NEXT open. Data gap/end -> last close.
    Returns (exit bar, gross return, exit type: 0 = filled at that bar's open, 1 = at/near that bar's close)."""
    T = C.shape[1]
    if e >= T: return -1, np.nan, 0
    fill = O[k, e]
    if not (fill > 0.0): return -1, np.nan, 0
    stop = fill * (1.0 - stop_pct)
    last = e + hold - 1
    if last > T - 1: last = T - 1
    for i in range(e, last + 1):
        c = C[k, i]
        if not (c > 0.0):
            if i == e: return -1, np.nan, 0
            return i - 1, C[k, i - 1] / fill - 1.0, 1
        if use_stop and L[k, i] <= stop:
            o = O[k, i]
            px = stop if (i == e or not (o > 0.0)) else (o if o < stop else stop)
            return i, px / fill - 1.0, 1
        if i == last:
            return i, c / fill - 1.0, 1
        hit = False
        if mode == 0:
            hit = c > S5[k, i]
        elif mode == 1:
            hit = RSI[k, i] > 70.0
        if hit:
            o = O[k, i + 1]
            if o > 0.0: return i + 1, o / fill - 1.0, 0
            return i, c / fill - 1.0, 1
    return last, C[k, last] / fill - 1.0, 1


@njit(cache=True)
def run_rule(sig, O, H, L, C, S5, RSI, mode, hold, use_stop, stop_pct, t_lo, t_hi):
    """Signals at close t in [t_lo, t_hi] -> entry t+1. One position per ticker (next signal allowed at t >= exit bar)."""
    N = C.shape[0]; cap = 0
    for k in range(N):
        for t in range(t_lo, t_hi + 1):
            if sig[k, t]: cap += 1
    ok = np.empty(cap, np.int64); ot = np.empty(cap, np.int64); ox = np.empty(cap, np.int64); orr = np.empty(cap)
    oxt = np.empty(cap, np.int64)
    m = 0
    for k in range(N):
        free = 0
        for t in range(t_lo, t_hi + 1):
            if sig[k, t] and t >= free:
                x, r, xt = sim_one(O, H, L, C, S5, RSI, k, t + 1, mode, hold, use_stop, stop_pct)
                if x >= 0:
                    ok[m] = k; ot[m] = t; ox[m] = x; orr[m] = r; oxt[m] = xt; m += 1; free = x
    return ok[:m], ot[:m], ox[:m], orr[:m], oxt[:m]


@njit(cache=True)
def sim_null(O, C, kk, e, x, xt):
    """Duration-matched null trade on ticker kk: enter at O[kk,e], exit at the matched real trade's exit bar x with
    the same price type (0: open, 1: close). Missing exit price -> last close in [e, x]; none -> rejected (-1)."""
    fill = O[kk, e]
    if not (fill > 0.0): return -1.0
    px = O[kk, x] if xt == 0 else C[kk, x]
    j = x
    while not (px > 0.0):
        if j < e: return -1.0
        px = C[kk, j]; j -= 1
    return px / fill - 1.0


@njit(parallel=True, cache=True)
def null_draws(tr_t, tr_k, tr_x, tr_xt, ptr, pidx, U, O, C, cost, yidx, ny):
    """SELECTION null (DURATION-MATCHED): for each real trade (signal bar t, ticker k, exit bar x, exit type) draw a
    random OTHER ticker from the pool at t (same universe, same non-trigger condition), enter at its next open and exit
    at the SAME bar with the same price type. Only the ticker differs, so holding-time x market-drift cannot bias it
    (the own-exit version failed the martingale protocol: S4 families at pct 64-66, t ~3, with drift on).
    Non-overlap per draw. Returns per-draw, per-entry-year sums and counts of net returns."""
    D = U.shape[0]; n = tr_t.shape[0]; tries = U.shape[2]; N = C.shape[0]
    S = np.zeros((D, ny)); Cn = np.zeros((D, ny)); miss = np.zeros(D, np.int64)
    for d in prange(D):
        busy = np.zeros(N, np.int64)
        for i in range(n):
            t = tr_t[i]; a = ptr[t]; m = ptr[t + 1] - a; done = False
            if m > 0:
                for j in range(tries):
                    q = int(U[d, i, j] * m)
                    if q >= m: q = m - 1
                    kk = pidx[a + q]
                    if kk == tr_k[i] or busy[kk] > t: continue
                    r = sim_null(O, C, kk, t + 1, tr_x[i], tr_xt[i])
                    if r == -1.0: continue
                    busy[kk] = tr_x[i]; y = yidx[t + 1]
                    if y >= 0 and y < ny:
                        S[d, y] += r - cost; Cn[d, y] += 1.0
                    done = True; break
            if not done: miss[d] += 1
    return S, Cn, miss


def year_ci(x, blocks, reps, seed, level):
    """Year-block bootstrap of the mean: (CI lo, CI hi, q = 100 x share of bootstrap means > 0). Stage 2b v2."""
    x = np.asarray(x, float); ub, inv = np.unique(blocks, return_inverse=True)
    Sm = np.bincount(inv, weights=x, minlength=len(ub)); Cn = np.bincount(inv, minlength=len(ub)).astype(float)
    idx = np.random.default_rng(seed).integers(0, len(ub), (reps, len(ub))); mb = Sm[idx].sum(1) / Cn[idx].sum(1)
    a = (1 - level) / 2
    return float(np.quantile(mb, a)), float(np.quantile(mb, 1 - a)), 100.0 * float((mb > 0).mean())


def run_cfg(ctx, cfg, u, n_draws, seed, fh_list):
    """Real trades of one rule in one universe + selection null + fixed-horizon excess."""
    U = ctx['UN'][u]
    if cfg['screen'] == 'S4':
        sig, pool = ctx['trend'] & ctx['TRIG'][cfg['trig']] & U, ctx['trend'] & U
    else:
        sig, pool = ctx['s5'] & U, ctx['s5pool'] & U
    nt = ctx['NT']; T = U.shape[0]
    k, t, x, r, xt = run_rule(np.ascontiguousarray(sig.T), nt['O'], nt['H'], nt['L'], nt['C'], nt['S5'], nt['RSI'],
                              cfg['mode'], cfg['hold'], cfg['stop'], S4_STOP, T_LO, T_HI)
    o = np.argsort(t, kind='stable'); k, t, x, r, xt = k[o], t[o], x[o], r[o], xt[o]
    net = r - COST; yr = YEAR_T[t + 1] if len(t) else np.array([], int)
    out = dict(cfg=cfg['name'], screen=cfg['screen'], family=cfg['family'], universe=u, n=len(t), k=k, t=t, x=x,
               gross=r, net=net, year=yr)
    # selection null
    tt, kk = np.nonzero(pool)
    ptr = np.searchsorted(tt, np.arange(T + 1)).astype(np.int64); pidx = kk.astype(np.int64)
    Sd, Cd = np.zeros((n_draws, NY)), np.zeros((n_draws, NY)); miss = 0; rng = np.random.default_rng(seed)
    for b0 in range(0, n_draws, DRAW_BATCH):
        nb_ = min(DRAW_BATCH, n_draws - b0)
        Ub = rng.random((nb_, len(t), TRIES), dtype=np.float32)
        s_, c_, m_ = null_draws(t, k, x, xt, ptr, pidx, Ub, nt['O'], nt['C'], COST, YIDX, NY)
        Sd[b0:b0 + nb_], Cd[b0:b0 + nb_] = s_, c_; miss += int(m_.sum())
    with np.errstate(invalid='ignore', divide='ignore'):
        dm = Sd.sum(1) / Cd.sum(1)
        out['null_year_mean'] = Sd.sum(0) / Cd.sum(0)
    out['null_mean'] = float(np.nanmean(dm)) if len(t) else np.nan
    out['pct'] = float(100 * np.mean(dm < net.mean())) if len(t) and np.isfinite(dm).all() else np.nan
    out['null_miss_rate'] = miss / max(1, n_draws * len(t))
    lo_, hi_ = ERAS[RECENT]; era = (yr >= lo_) & (yr <= hi_)
    ey = np.arange(lo_ - Y0, hi_ - Y0 + 1)
    with np.errstate(invalid='ignore', divide='ignore'):
        out['era_exc'] = (float(net[era].mean() - Sd[:, ey].sum() / Cd[:, ey].sum()) if era.any() else np.nan)
    # fixed-horizon excess: close of the fill bar -> close h bars later, vs universe EW and SPY, same window
    out['fh'] = {}
    for h in fh_list:
        if not len(t): out['fh'][h] = None; continue
        e = t + 1; F = ctx['FW'][h]
        s_ = F[e, k]; ew = ctx['EWF'][(u, h)][e]; sp = F[e, ctx['i_spy']]
        res = {}
        for bm, b in (('ew', ew), ('spy', sp)):
            d_ = s_ - b; ok_ = np.isfinite(d_)
            ci = year_ci(d_[ok_], yr[ok_], BOOT_REPS, seed + h, CI_LEVEL) if ok_.sum() >= 20 else (np.nan,) * 3
            res[bm] = dict(n=int(ok_.sum()), mean=float(d_[ok_].mean()) if ok_.any() else np.nan,
                           lo=ci[0], hi=ci[1], q=ci[2])
        out['fh'][h] = res
    return out


def s6_monthly(ctx, u, cal):
    """Month-end formation: top decile (>= 5 names) by trailing 21-bar overnight-return sum; hold close->next
    month-end close, EW; minus all universe members over the same window. Turnover cost = replaced share x COST."""
    C, ON21, Um = ctx['C'], ctx['I']['ON21'], ctx['UN'][u]
    per = cal.to_period('M'); me = list(np.nonzero(per[:-1] != per[1:])[0]) + [len(cal) - 1]
    Cff = pd.DataFrame(C).ffill().to_numpy(); rows, prev = [], None
    for a, b in zip(me[:-1], me[1:]):
        if cal[b] < IS_START or cal[b] > PURGE: continue
        idx = np.nonzero(Um[a] & np.isfinite(ON21[a]) & (C[a] > 0))[0]
        if len(idx) < 2 * S6_MIN_NAMES: continue
        kk = max(int(math.ceil(len(idx) / 10)), S6_MIN_NAMES)
        top = idx[np.argsort(-ON21[a, idx], kind='stable')[:kk]]
        R_all = Cff[b, idx] / C[a, idx] - 1; R_top = Cff[b, top] / C[a, top] - 1
        gross = float(np.nanmean(R_top) - np.nanmean(R_all))
        turn = 1.0 if prev is None else 1 - len(set(top.tolist()) & prev) / kk; prev = set(top.tolist())
        rows.append(dict(month=str(cal[b].to_period('M')), year=cal[b].year, n_univ=len(idx), k=kk, gross=gross,
                         turnover=turn, net=gross - turn * COST))
    return pd.DataFrame(rows)


def s6_summary(M, seed=0):
    if M.empty: return dict(n=0)
    lo_, hi_ = ERAS[RECENT]; era = M[(M.year >= lo_) & (M.year <= hi_)]
    ci = year_ci(M['net'].to_numpy(), M['year'].to_numpy(), BOOT_REPS, seed, CI_LEVEL) if len(M) >= 20 else (np.nan,) * 3
    cig = year_ci(M['gross'].to_numpy(), M['year'].to_numpy(), BOOT_REPS, seed, CI_LEVEL) if len(M) >= 20 else (np.nan,) * 3
    yr = M.groupby('year')['net'].sum().reindex(range(Y0, Y0 + NY))
    return dict(n=len(M), mean_gross=M.gross.mean(), mean_turn=M.turnover.mean(), mean_net=M.net.mean(),
                ci_lo=ci[0], ci_hi=ci[1], q=ci[2], ci_lo_gross=cig[0], era_net=era.net.mean() if len(era) else np.nan,
                yrs_pos=int((yr > 0).sum()), breakeven=(M.gross.mean() / M.turnover.mean()) if M.turnover.mean() > 0 else np.nan,
                per_year=yr)


def calendar_consts(cal):
    global T_LO, T_HI, YEAR_T, YIDX
    T_LO = int(np.searchsorted(cal, IS_START)) - 1                 # signal bar; entry bar = T_LO + 1 >= IS_START
    T_HI = int(np.searchsorted(cal, PURGE, side='right')) - 2      # entry bar T_HI + 1 <= PURGE
    assert T_LO >= 0 and cal[T_LO + 1] >= IS_START and cal[T_HI + 1] <= PURGE
    YEAR_T = cal.year.to_numpy()
    YIDX = np.where((YEAR_T >= Y0) & (YEAR_T < Y0 + NY), YEAR_T - Y0, -1).astype(np.int64)


def read_assets():
    """ASSET_FILE with the reviewed ASSET_OVERRIDES applied."""
    A = pd.read_csv(ASSET_FILE).fillna({'name': ''})
    A['asset_class'] = [ASSET_OVERRIDES.get(t, c) for t, c in zip(A['ticker'], A['asset_class'])]
    return A


def asset_masks(tickers):
    A = read_assets().set_index('ticker').reindex(tickers)
    u1 = set(pd.read_csv(U1_FILE)['ticker'])
    is_u1 = np.array([t in u1 for t in tickers])
    is_stock = A['asset_class'].isin(['Stock', 'Unknown']).to_numpy() & ~is_u1
    return is_stock, is_u1


def sha(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()



# ---------------------------------------------------------------- S4b event study
def etf_ctx(O, H, L, C, V):
    """Indicators, trend and triggers on the 12-ETF panel (T x 12); N x T copies for the simulator."""
    I = indicators(O, H, L, C, V)
    C1, C2, C3 = lag(C, 1), lag(C, 2), lag(C, 3)
    trend = (C > I['S200']) & (I['S50'] > I['S200'])
    TRIG = {'T1': I['RSI'] < 10, 'T2': (C < C1) & (C1 < C2) & (C2 < C3), 'T3': C < I['LOW20p']}
    NT = {k: np.ascontiguousarray(x.T) for k, x in dict(O=O, H=H, L=L, C=C, S5=I['S5'], RSI=I['RSI']).items()}
    return dict(trend=trend, TRIG=TRIG, NT=NT, O=O, C=C)


def baseline_table(ctx, trig, max_len=20):
    """Mean return over every other uptrend day of the same ETF (trend true, THIS trigger false; signal day in the IS
    range), entry at the next open, exit L bars later at the open (xt 0) or close (xt 1). Shape (N, max_len+1, 2)."""
    O, C = ctx['O'], ctx['C']; T, N = C.shape
    B = ctx['trend'] & ~ctx['TRIG'][trig]
    out = np.full((N, max_len + 1, 2), np.nan)
    for k in range(N):
        s = np.nonzero(B[T_LO:T_HI + 1, k])[0] + T_LO; e = s + 1
        for L_ in range(max_len + 1):
            x = e + L_; ok = x < T
            for xt, PX in ((0, O), (1, C)):
                r = PX[x[ok], k] / O[e[ok], k] - 1
                out[k, L_, xt] = np.nanmean(r) if np.isfinite(r).any() else np.nan
    return out


def run_trial(ctx, trig, outcome, seed=0):
    """Non-overlapping events per ETF (the simulator's one-position rule), excess = event gross - same-ETF baseline of
    the SAME length and exit price type - COST (costs on the event only; baseline gross)."""
    name, mode, hold = outcome
    sig = ctx['trend'] & ctx['TRIG'][trig]; nt = ctx['NT']
    k, t, x, r, xt = run_rule(np.ascontiguousarray(sig.T), nt['O'], nt['H'], nt['L'], nt['C'], nt['S5'], nt['RSI'],
                              mode, hold, False, 0.0, T_LO, T_HI)
    BT = baseline_table(ctx, trig)
    Lr = x - (t + 1)
    base = BT[k, Lr, xt]
    gross_exc = r - base; exc = gross_exc - COST
    ok = np.isfinite(exc); k, t, x, r, exc, gross_exc = k[ok], t[ok], x[ok], r[ok], exc[ok], gross_exc[ok]
    ent = CAL[t + 1]; yr = ent.year.to_numpy(); mo = np.asarray(ent.to_period('M').astype(str))
    out = dict(trig=trig, outcome=name, n=len(exc), k=k, t=t, exc=exc, year=yr, month=mo)
    if len(exc) < 20:
        out.update(mean=np.nan, q=np.nan, mean_gross=np.nan, q_gross=np.nan, verdict=False); return out
    lo_, hi_, q_ = year_ci(exc, mo, BOOT_REPS, seed, CI_LEVEL)          # month-block bootstrap (NET: the verdict)
    q_g = year_ci(gross_exc, mo, BOOT_REPS, seed, CI_LEVEL)[2]          # GROSS: what the martingale check centres
    yt = pd.DataFrame({'year': yr, 'exc': exc}).groupby('year')['exc'].agg(['size', 'mean']).reindex(range(Y0, Y0 + NY))
    qual = yt[yt['size'] >= MIN_EVENTS_YEAR]
    n_q, n_pos = len(qual), int((qual['mean'] > 0).sum())
    lo_e, hi_e = ERAS[RECENT]; era = (yr >= lo_e) & (yr <= hi_e)
    era_m = float(exc[era].mean()) if era.any() else np.nan
    verdict = bool(np.isfinite(era_m) and era_m > 0 and n_q >= MIN_QUAL_YEARS and n_pos >= math.ceil(2 * n_q / 3)
                   and q_ >= Q_PASS)
    out.update(mean=float(exc.mean()), mean_gross=float(gross_exc.mean()), q_gross=q_g, breakeven=float(gross_exc.mean()),
               avg_gross=float(r.mean()), ci_lo=lo_, ci_hi=hi_, q=q_, era=era_m, n_qual=n_q, n_pos_qual=n_pos,
               avg_bars=float((x - t).mean()), per_year=yt, verdict=verdict)
    return out


@njit(cache=True)
def gen_world(seed, mkt, sd_i, first, last, P0, M, on_frac, plant_rev, rev_thr, plant_vol, vspike):
    """Pack-2 generator (intrabar paths, martingale idio, market drift); copied verbatim."""
    np.random.seed(seed)
    T = mkt.shape[0]; N = sd_i.shape[0]
    O = np.full((N, T), np.nan); Hh = np.full((N, T), np.nan); Ll = np.full((N, T), np.nan); Cc = np.full((N, T), np.nan)
    for k in range(N):
        if first[k] < 0: continue
        so = sd_i[k] * np.sqrt(on_frac); si = sd_i[k] * np.sqrt((1.0 - on_frac) / M)
        prev = P0[k]; cr = 0; cv = 0
        for t in range(first[k], last[k] + 1):
            drift = 0.0
            if cr > 0:
                drift += plant_rev; cr -= 1
            if cv > 0:
                drift += plant_vol; cv -= 1
            o = prev * np.exp(mkt[t, 0] + so * np.random.standard_normal() - 0.5 * so * so)
            p = o; hi = o; lo = o
            for j in range(1, M + 1):
                p *= np.exp(mkt[t, j] + si * np.random.standard_normal() - 0.5 * si * si + drift / M)
                if p > hi: hi = p
                if p < lo: lo = p
            O[k, t] = o; Hh[k, t] = hi; Ll[k, t] = lo; Cc[k, t] = p
            if plant_rev > 0.0 and np.log(p / prev) < -rev_thr[k]: cr = 3
            if plant_vol > 0.0 and vspike[k, t]: cv = 20
            prev = p
    return O, Hh, Ll, Cc


def append_log(path, df, key='config_id'):
    """Append-only; an identical config already logged is not a new trial."""
    if path.exists():
        old = pd.read_csv(path, dtype={key: str}); new = df[~df[key].isin(old[key])]
        pd.concat([old, new], ignore_index=True).to_csv(path, index=False); return len(new), len(old) + len(new)
    df.to_csv(path, index=False); return len(df), len(df)

In [ ]:
# ===== Cell 2 — panels (cut at load; pack-2 cache) -> the 12 ETFs =====
t0 = time.time()
TICKERS, CAL, P = load_panels(); calendar_consts(CAL)
missing = [e for e in ETFS if e not in TICKERS]; assert not missing, f'ETFs missing from the corpus: {missing}'
IX = [TICKERS.index(e) for e in ETFS]
E = {c: np.ascontiguousarray(P[c][:, IX]) for c in 'OHLCV'}; del P
assert CAL.max() <= CUT_DATE
first_bar = {e: CAL[int(np.argmax(np.isfinite(E['C'][:, i])))].date() for i, e in enumerate(ETFS)}
print(f'{len(ETFS)} ETFs x {len(CAL)} bars [{time.time()-t0:.0f}s]; first bars: {first_bar}')

## Part A — martingale validation (must PASS before Part B)

In [ ]:
# ===== Cell 3 — worlds restricted to the 12 ETFs =====
Creal = E['C']; T_, N_ = Creal.shape; valid = np.isfinite(Creal)
FIRST = np.where(valid.any(0), valid.argmax(0), -1).astype(np.int64)
LAST = np.where(valid.any(0), T_ - 1 - valid[::-1].argmax(0), -1).astype(np.int64)
P0 = np.array([Creal[FIRST[i], i] for i in range(N_)])
sd = np.clip(np.nanstd(np.diff(np.log(Creal), axis=0), axis=0), 0.005, 0.10)
SD_I = np.sqrt(np.maximum(sd ** 2 - MKT_SD ** 2, (0.25 * sd) ** 2))
NOSPIKE = np.zeros((N_, T_), bool)


def make_world(seed, plant=False):
    rng = np.random.default_rng(20_000 + seed)
    mkt = np.empty((T_, M_STEPS + 1))
    mkt[:, 0] = rng.normal(MKT_MU * ON_FRAC - 0.5 * MKT_SD ** 2 * ON_FRAC, MKT_SD * np.sqrt(ON_FRAC), T_)
    s_in = MKT_SD * np.sqrt((1 - ON_FRAC) / M_STEPS)
    mkt[:, 1:] = rng.normal(MKT_MU * (1 - ON_FRAC) / M_STEPS - 0.5 * s_in ** 2, s_in, (T_, M_STEPS))
    O, Hh, Ll, Cc = gen_world(seed, mkt, SD_I, FIRST, LAST, P0, M_STEPS, ON_FRAC,
                              PLANT_REV if plant else 0.0, 2 * sd, 0.0, NOSPIKE)
    W = {c: np.ascontiguousarray(x.T) for c, x in zip('OHLC', (O, Hh, Ll, Cc))}
    for c in 'OHLC': W[c][~valid] = np.nan
    return W


def run_world(seed, plant=False, trigs=TRIGS):
    W = make_world(seed, plant); ctx = etf_ctx(W['O'], W['H'], W['L'], W['C'], E['V'])
    return [dict(seed=seed, trig=tg, outcome=oc[0], **{k: v for k, v in run_trial(ctx, tg, oc, seed).items()
                                                      if k in ('n', 'mean_gross', 'q_gross')})
            for tg in trigs for oc in OUTCOMES]


REC = []
for seed in range(S4B_SEEDS):
    t0 = time.time(); r_ = run_world(seed); REC += r_; d_ = pd.DataFrame(r_)
    print(f'seed {seed:2d}: {time.time()-t0:4.0f}s | events/trial median {int(d_.n.median())} | '
          f'q_gross>=90 {int((d_.q_gross >= Q_PASS).sum())}/15 | mean q_gross {d_.q_gross.mean():.1f}')
REC = pd.DataFrame(REC); REC.to_csv(S4B_DIR/'protocol_records.csv', index=False)

In [ ]:
# ===== Cell 4 — evaluate + power + write s4b_protocol.json =====
def t_vs(v, mu):
    v = np.asarray(v, float); v = v[np.isfinite(v)]
    return float(stats.ttest_1samp(v, mu).statistic) if len(v) > 2 and v.std() > 0 else np.nan

fam = {}
for tg, g in REC.groupby('trig'):
    ps = g.groupby('seed').agg(mean=('mean_gross', 'mean'), q=('q_gross', 'mean'))
    fam[tg] = dict(mean_gross_exc=ps['mean'].mean(), t_exc=t_vs(ps['mean'], 0.0), mean_q=ps['q'].mean(),
                   t_q=t_vs(ps['q'], 50.0), pass_rate=float((g.q_gross >= Q_PASS).mean()), n_seeds=len(ps))
FAM = pd.DataFrame(fam).T
pooled = float((REC.q_gross >= Q_PASS).mean())
centring = bool(len(FAM) == 3 and (FAM.t_exc.abs() < 2.5).all() and FAM.mean_q.between(40, 60).all()
                and (FAM.t_q.abs() < 2.5).all())
NULL_OK = bool(centring and pooled <= 0.15)
print(FAM.round(4).to_string())
print(f'GROSS excess (cost is a known constant, so the null is centred gross; the verdict on real data is NET)')
print(f'pooled share q_gross >= {Q_PASS:.0f}: {pooled:.1%} (<= 15%) | centring {"PASS" if centring else "FAIL"}')
print('by outcome:'); print(REC.groupby('outcome').agg(mean_gross=('mean_gross', 'mean'), q_gross=('q_gross', 'mean')).round(4).to_string())

POWER_OK, det = False, []
if NULL_OK:
    for s in range(S4B_POWER_SEEDS):
        pw = pd.DataFrame(run_world(1000 + s, plant=True, trigs=['T1']))
        det.append(float(pw.q_gross.mean())); print(f'power seed {1000+s}: T1 mean q_gross {det[-1]:.1f}')
    POWER_OK = sum(d >= Q_PASS for d in det) >= math.ceil(0.75 * S4B_POWER_SEEDS)
S4B_OK = bool(NULL_OK and POWER_OK)
json.dump(dict(written_utc=pd.Timestamp.now(tz='UTC').isoformat(), n_seeds=S4B_SEEDS, power_seeds=S4B_POWER_SEEDS,
               families=json.loads(FAM.to_json(orient='index')), pooled=pooled, centring=centring, null_ok=NULL_OK,
               power_q=det, power_ok=POWER_OK, ok=S4B_OK), open(S4B_PROTO, 'w'), indent=1, default=float)
print(f'SUMMARY S4b protocol ({S4B_SEEDS} seeds): null {"PASS" if NULL_OK else "FAIL"} | power '
      f'{"PASS" if POWER_OK else ("FAIL" if NULL_OK else "not run")} -> real data {"ALLOWED" if S4B_OK else "STOPPED"}')

## Part B — real data (gated on Part A)

In [ ]:
# ===== Cell 5 — gate =====
PRO = json.load(open(S4B_PROTO))
assert PRO['ok'], 'S4b martingale validation did not pass -> STOP (standing rule); real data not touched'
assert PRO['n_seeds'] >= 16 or os.environ.get('PACK2_TEST') == '1', 'needs the 16-seed validation'
print('S4b protocol PASS', PRO['written_utc'])

In [ ]:
# ===== Cell 6 — 15 trials on real data =====
assert json.load(open(S4B_PROTO))['ok'], 'S4b validation did not pass -> STOP'
CTX = etf_ctx(E['O'], E['H'], E['L'], E['C'], E['V'])
RES = {(tg, oc[0]): run_trial(CTX, tg, oc, 31 + i) for i, (tg, oc) in enumerate((tg, oc) for tg in TRIGS for oc in OUTCOMES)}
pct = lambda v: f'{v:+.2%}' if np.isfinite(v) else 'nan'
rows = {}
for key, o in RES.items():
    rows[key] = dict(n=o['n'], avg_bars=o.get('avg_bars'), avg_gross=pct(o.get('avg_gross', np.nan)),
                     excess_net=pct(o['mean']), ci90=f'[{pct(o.get("ci_lo", np.nan))}, {pct(o.get("ci_hi", np.nan))}]',
                     q=round(o['q'], 1) if np.isfinite(o['q']) else np.nan, era_19_23=pct(o.get('era', np.nan)),
                     qual_years=o.get('n_qual'), pos_qual=o.get('n_pos_qual'), breakeven=pct(o.get('breakeven', np.nan)),
                     VERDICT=o['verdict'])
TB = pd.DataFrame(rows).T; TB.index.names = ['trigger', 'outcome']
pd.set_option('display.width', 250)
print('S4b — pooled over 12 ETFs (excess = event - same-length same-ETF baseline - 0.15%):'); print(TB.to_string())
PY = pd.DataFrame({f'{k[0]}_{k[1]}': o['per_year']['mean'] for k, o in RES.items() if 'per_year' in o}).T * 1e4
PN = pd.DataFrame({f'{k[0]}_{k[1]}': o['per_year']['size'] for k, o in RES.items() if 'per_year' in o}).T
print('\nPer-year mean excess (bp):'); print(PY.round(0).to_string())
print('\nPer-year event counts (a year qualifies with >= 10):'); print(PN.fillna(0).astype(int).to_string())
PE = pd.DataFrame({f'{k[0]}_{k[1]}': pd.Series(o['exc']).groupby(np.asarray(ETFS)[o['k']]).mean()
                   for k, o in RES.items() if o['n']}).T * 1e4
print('\nPer-ETF mean excess (bp, descriptive):'); print(PE.round(0).to_string())
TB.to_csv(S4B_DIR/'s4b_trials.csv'); PY.to_csv(S4B_DIR/'s4b_per_year_bp.csv'); PE.to_csv(S4B_DIR/'s4b_per_etf_bp.csv')

In [ ]:
# ===== Cell 7 — verdicts + trial log =====
assert json.load(open(S4B_PROTO))['ok'], 'S4b validation did not pass -> STOP'
now = pd.Timestamp.now(tz='UTC').isoformat(); rows, rets = [], []
for (tg, oc), o in RES.items():
    cid = f's4b|S4b|ETF12|{tg}_{oc}'
    m = pd.Series(o['exc']).groupby(o['month']).mean() if o['n'] else pd.Series(dtype=float)
    rows.append(dict(kind='trial', config_id=cid, pack='s4b', screen='S4b', universe='ETF12', rule=f'{tg}_{oc}',
                     n=o['n'], n_months=len(m), WR=float((o['exc'] > 0).mean()) if o['n'] else np.nan,
                     avg_net=o['mean'], sd=m.std(), skew=m.skew(), kurt=m.kurt(),
                     sr_monthly=m.mean() / m.std() if m.std() > 0 else np.nan, pct_null=o['q'], verdict=o['verdict'],
                     backfilled=False, note='excess vs same-length same-ETF baseline, net of 0.15%', logged_at=now))
    rets += [dict(config_id=cid, month=mm, ret=v) for mm, v in m.items()]
LOG = pd.DataFrame(rows); RET = pd.DataFrame(rets)
if len(RET): RET['key'] = RET.config_id + '|' + RET.month
n_new, n_tot = append_log(TRIAL_LOG, LOG)
if len(RET): append_log(TRIAL_RET, RET, key='key')
allL = pd.read_csv(TRIAL_LOG)
print(f'trial log: +{n_new} new | cumulative trials {int((allL.kind == "trial").sum())}')
print('\nVERDICTS (era 2019-2023 > 0 AND >= 9 qualifying years with >= 2/3 positive AND q >= 90):')
for _, r in LOG.iterrows():
    print(f'  {"WORTH DRILLING    " if r.verdict else "not worth drilling"}  {r.rule:8s}  q {r.pct_null:5.1f}  '
          f'excess {r.avg_net:+.3%}')